# 04 — Reporting Views: Fleet & Route Performance Analytics
Creates SQL views over Gold tables for Power BI and ad-hoc analysis.

| View | Purpose |
|---|---|
| vw_executive_summary | Single-row KPI snapshot |
| vw_late_deliveries | Operational late-delivery feed |
| vw_fuel_analysis | Depot-level fuel cost and efficiency |
| vw_route_efficiency | Route performance ranked by on-time rate |

In [ ]:
LAKEHOUSE_NAME = "transportation_lakehouse"
GOLD = "gold"
VIEWS = "reporting"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {LAKEHOUSE_NAME}.{VIEWS}")
print("Reporting schema ready.")

In [ ]:
# ── vw_executive_summary ─────────────────────────────────────────────────────
spark.sql(f"""
CREATE OR REPLACE VIEW {LAKEHOUSE_NAME}.{VIEWS}.vw_executive_summary AS
SELECT
    SUM(total_deliveries)                                    AS total_deliveries,
    SUM(late_deliveries)                                     AS total_late_deliveries,
    ROUND(AVG(on_time_rate_pct), 1)                          AS avg_on_time_rate_pct,
    ROUND(AVG(avg_delay_hrs), 2)                             AS avg_delay_hrs,
    SUM(total_km_driven)                                     AS total_km_driven,
    ROUND(AVG(avg_load_utilisation_pct), 1)                  AS avg_load_utilisation_pct
FROM {LAKEHOUSE_NAME}.{GOLD}.gold_fleet_summary
""")
print("vw_executive_summary created")

In [ ]:
# ── vw_late_deliveries ───────────────────────────────────────────────────────
spark.sql(f"""
CREATE OR REPLACE VIEW {LAKEHOUSE_NAME}.{VIEWS}.vw_late_deliveries AS
SELECT
    delivery_id,
    vehicle_id,
    origin,
    destination,
    planned_departure,
    actual_arrival,
    delay_hrs,
    delay_band
FROM {LAKEHOUSE_NAME}.{GOLD}.gold_late_delivery_alerts
ORDER BY delay_hrs DESC
""")
print("vw_late_deliveries created")

In [ ]:
# ── vw_fuel_analysis ─────────────────────────────────────────────────────────
spark.sql(f"""
CREATE OR REPLACE VIEW {LAKEHOUSE_NAME}.{VIEWS}.vw_fuel_analysis AS
SELECT
    depot,
    fleet_size,
    active_vehicles,
    total_fuel_cost_gbp,
    avg_fuel_efficiency,
    ROUND(total_fuel_cost_gbp / NULLIF(fleet_size, 0), 2)   AS fuel_cost_per_vehicle_gbp
FROM {LAKEHOUSE_NAME}.{GOLD}.gold_depot_scorecards
ORDER BY total_fuel_cost_gbp DESC
""")
print("vw_fuel_analysis created")

In [ ]:
# ── vw_route_efficiency ──────────────────────────────────────────────────────
spark.sql(f"""
CREATE OR REPLACE VIEW {LAKEHOUSE_NAME}.{VIEWS}.vw_route_efficiency AS
SELECT
    route_id,
    origin,
    destination,
    route_type,
    total_deliveries,
    on_time_rate_pct,
    avg_delay_hrs,
    avg_load_pct,
    total_toll_cost_gbp
FROM {LAKEHOUSE_NAME}.{GOLD}.gold_route_analysis
ORDER BY on_time_rate_pct ASC
""")
print("vw_route_efficiency created")

In [ ]:
print("\n=== Reporting Views Complete ===")
for v in ["vw_executive_summary", "vw_late_deliveries", "vw_fuel_analysis", "vw_route_efficiency"]:
    spark.sql(f"SELECT * FROM {LAKEHOUSE_NAME}.{VIEWS}.{v}").show(3, truncate=False)